# Anggota 6 — Penilaian Jawaban Akhir

**Tugas Akhir Pemrosesan Bahasa Alami · QA factoid tanpa LLM**

Notebook ini menilai jawaban akhir sistem: lima kandidat passage dari Anggota 5 diolah ekstraktor Anggota 4, lalu jawabannya dibandingkan dengan jawaban rujukan. Hasilnya akurasi jawaban, bukan lagi metrik retrieval.

Jalankan notebook dari folder `Anggota6_Evaluasi_Jawaban` dengan kernel Python 3.11+ dari lingkungan `.venv`, lalu pilih **Run All**. Model UDPipe Indonesia diunduh pada pemakaian pertama. Petunjuk instalasi ada di [README](README.md).

## 1. Membaca hasil retrieval dan metadata korpus

Masukan satu-satunya adalah [keluaran Anggota 5](../Anggota5_Evaluasi_Retrieval/output/hasil_retrieval.json) yang sudah memuat pertanyaan, jawaban rujukan, peringkat bukti, dan `kandidat_top5`. Metadata dokumen dipakai untuk menampilkan sumber jawaban.

In [1]:
import json
import sys

from IPython.display import Markdown, display

sys.path.insert(0, "..")

from Anggota6_Evaluasi_Jawaban.evaluasi_jawaban import (
    AMBANG_F1,
    evaluasi,
    putusan_jawaban,
)

with open("../Anggota5_Evaluasi_Retrieval/output/hasil_retrieval.json", encoding="utf-8") as file:
    hasil_retrieval = json.load(file)
with open("../Anggota1_Korpus/data/dokumen.json", encoding="utf-8") as file:
    dokumen = json.load(file)

soal = hasil_retrieval["hasil_per_pertanyaan"]
print(f"Jumlah soal: {len(soal)}")
print(f"Metrik retrieval Anggota 5: Hit@5 {hasil_retrieval['ringkasan']['hit_at_5']:.1%}, "
      f"MRR {hasil_retrieval['ringkasan']['mrr']:.3f}")

Jumlah soal: 20
Metrik retrieval Anggota 5: Hit@5 85.0%, MRR 0.780


## 2. Aturan penilaian

Penilaian tidak memakai LLM. Jawaban sistem dan jawaban rujukan dinormalisasi (Unicode NFKC, case folding, penanda butir dibuang, spasi dan tanda baca tepi dirapikan), lalu dibandingkan pada empat tingkat:

| Tingkat | Arti |
| --- | --- |
| `COCOK_PERSIS` | Sama setelah normalisasi. |
| `COCOK_INTI` | Salah satu jawaban memuat jawaban lain sebagai deret token utuh, misalnya `14 semester` di dalam `14 semester sejak pertama kali terdaftar …`. |
| `MIRIP_PARAFRASA` | F1 bag-of-token ≥ ambang, dipakai untuk jawaban benar yang disusun berbeda. |
| `TIDAK_COCOK` | Di bawah semua ambang. |

Akurasi ketat memakai `COCOK_PERSIS`, akurasi inti menambahkan `COCOK_INTI`, dan akurasi longgar menambahkan `MIRIP_PARAFRASA`. Jawaban benar yang memuat kualifikasi tambahan tetap terhitung benar, sehingga angkanya harus dibaca bersama tabel per soal.

In [2]:
contoh = [
    ("144 SKS.", "144 sks"),
    ("14 semester", "14 semester sejak pertama kali terdaftar sebagai mahasiswa."),
    ("Paling lambat 1 bulan setelah berakhirnya masa daftar ulang.",
     "Paling lambat 1 bulan setelah masa daftar ulang berakhir."),
    ("120 sks", "138 sks"),
    ("Format skripsi disusun menurut dokumen Panduan Skripsi.",
     "Karya ilmiah mahasiswa pendidikan program sarjana yang merupakan wujud dari kajian pengetahuan."),
]
for jawaban_sistem, jawaban_rujukan in contoh:
    putusan = putusan_jawaban(jawaban_sistem, jawaban_rujukan)
    print(f'{putusan["tingkat"]:<16} F1 {putusan["skor_f1"]:.3f} · {jawaban_sistem} ↔ {jawaban_rujukan}')

assert putusan_jawaban("144 SKS.", "144 sks")["tingkat"] == "COCOK_PERSIS"
assert putusan_jawaban("14 semester", "14 semester sejak pertama kali terdaftar.")["tingkat"] == "COCOK_INTI"
assert putusan_jawaban("120 sks", "138 sks")["tingkat"] == "TIDAK_COCOK"
print(f"\nAmbang F1 parafrasa: {AMBANG_F1}")

COCOK_PERSIS     F1 1.000 · 144 SKS. ↔ 144 sks
COCOK_INTI       F1 0.400 · 14 semester ↔ 14 semester sejak pertama kali terdaftar sebagai mahasiswa.
MIRIP_PARAFRASA  F1 0.889 · Paling lambat 1 bulan setelah berakhirnya masa daftar ulang. ↔ Paling lambat 1 bulan setelah masa daftar ulang berakhir.
TIDAK_COCOK      F1 0.500 · 120 sks ↔ 138 sks
TIDAK_COCOK      F1 0.000 · Format skripsi disusun menurut dokumen Panduan Skripsi. ↔ Karya ilmiah mahasiswa pendidikan program sarjana yang merupakan wujud dari kajian pengetahuan.

Ambang F1 parafrasa: 0.7


## 3. Menjalankan ekstraksi Anggota 4 dan menilai seluruh soal

Setiap baris Anggota 5 diolah dengan `proses_pertanyaan` dari Anggota 2, lalu kelima kandidat diteruskan ke `EkstraktorJawaban.ekstrak`. Ekstraktor memilih jalur simbolik, linguistik UDPipe, atau fallback kalimat sendiri; modul ini tidak menambah aturan jawaban.

In [3]:
hasil = evaluasi(hasil_retrieval, dokumen)
ringkasan = hasil["ringkasan"]
baris_hasil = hasil["hasil_per_pertanyaan"]

display(Markdown(
    f'**Akurasi ketat:** {ringkasan["jumlah_persis"]}/{ringkasan["jumlah_pertanyaan"]} '
    f'({ringkasan["akurasi_persis"]:.1%}) · '
    f'**Akurasi inti:** {ringkasan["jumlah_cocok_inti"]}/{ringkasan["jumlah_pertanyaan"]} '
    f'({ringkasan["akurasi_inti"]:.1%}) · '
    f'**Akurasi longgar:** {ringkasan["jumlah_benar_longgar"]}/{ringkasan["jumlah_pertanyaan"]} '
    f'({ringkasan["akurasi_longgar"]:.1%}) · '
    f'**F1 token macro:** {ringkasan["f1_token_macro"]:.3f}'
))

def pendek(teks, batas=70):
    teks = " ".join(str(teks).split()).replace("|", r"\|")
    return teks if len(teks) <= batas else teks[: batas - 1] + "…"

tabel = [
    "| ID | Tipe | Bukti | Peringkat | Jawaban sistem | Jawaban rujukan | Putusan |",
    "| --- | --- | --- | ---: | --- | --- | --- |",
]
for baris in baris_hasil:
    metode = baris["metode"].replace("_", " ")
    tabel.append(
        f'| {baris["id_pertanyaan"]} | {baris["tipe_jawaban"]} | {metode} '
        f'| {baris["peringkat_bukti_pertama"] or "—"} '
        f'| {pendek(baris["jawaban_sistem"])} | {pendek(baris["jawaban_rujukan"])} '
        f'| {baris["tingkat"]} |'
    )
display(Markdown("\n".join(tabel)))

**Akurasi ketat:** 8/20 (40.0%) · **Akurasi inti:** 13/20 (65.0%) · **Akurasi longgar:** 15/20 (75.0%) · **F1 token macro:** 0.696

| ID | Tipe | Bukti | Peringkat | Jawaban sistem | Jawaban rujukan | Putusan |
| --- | --- | --- | ---: | --- | --- | --- |
| A2-001 | ANGKA | linguistik udpipe | 3 | 144 sks | 144 sks | COCOK_PERSIS |
| A2-002 | DURASI | linguistik udpipe | 1 | 75 menit | 75 menit | COCOK_PERSIS |
| A2-003 | DEFINISI | fallback kalimat | 19 | Format skripsi disusun menurut dokumen Panduan Skripsi. | Karya ilmiah mahasiswa pendidikan program sarjana yang merupakan wuju… | TIDAK_COCOK |
| A2-004 | NILAI_HURUF | fallback kalimat | 6 | Menurut pedoman penilaian PTIIK, nilai angka (N) dengan rentang 80 < … | B+ | TIDAK_COCOK |
| A3-001 | DURASI | linguistik udpipe | 2 | 14 semester | 14 semester sejak pertama kali terdaftar sebagai mahasiswa. | COCOK_INTI |
| A3-002 | WAKTU | linguistik udpipe | 1 | Paling lambat 1 bulan setelah berakhirnya masa daftar ulang. | Paling lambat 1 bulan setelah masa daftar ulang berakhir. | MIRIP_PARAFRASA |
| A3-003 | RUMUS | fallback kalimat | 25 | Keberhasilan studi mahasiswa dinyatakan dengan Indeks Prestasi (IP). | IP = Σ (K_i × Na_i) / Σ K_i. | TIDAK_COCOK |
| A4-001 | PERSENTASE | linguistik udpipe | 1 | 80% | 80% | COCOK_PERSIS |
| A4-002 | DURASI | linguistik udpipe | 1 | 15 menit | 15 menit | COCOK_PERSIS |
| A4-003 | ANGKA | simbolik | 1 | 3,5 | 3,5 | COCOK_PERSIS |
| A4-004 | JABATAN | linguistik udpipe | 2 | Dekan | Dekan | COCOK_PERSIS |
| FARIS-001 | DURASI | linguistik udpipe | 1 | 2 semester | 2 semester kumulatif | COCOK_INTI |
| FARIS-002 | DURASI | linguistik udpipe | 1 | 2 tahun | 2 tahun kumulatif | COCOK_INTI |
| FARIS-003 | ANGKA | linguistik udpipe | 1 | 120 sks | 138 sks | TIDAK_COCOK |
| FARIS-004 | DURASI | linguistik udpipe | 1 | 30 hari | 30 hari kalender | COCOK_INTI |
| SUP-001 | WAKTU | linguistik udpipe | 1 | 1 minggu setelah pelaksanaan UTS atau UAS. | Maksimal satu minggu setelah pelaksanaan UTS atau UAS. | MIRIP_PARAFRASA |
| SUP-002 | ANGKA | linguistik udpipe | 1 | 48 sks | 48 sks lulus | COCOK_INTI |
| SUP-003 | NILAI_HURUF | fallback kalimat | 1 | Ujian khusus diberikan kepada mahasiswa yang akan Ujian Akhir Sarjana… | C | TIDAK_COCOK |
| SUP-004 | DURASI | linguistik udpipe | 1 | 2 semester | 2 semester | COCOK_PERSIS |
| SUP-005 | ANGKA | simbolik | 1 | 24 sks | 24 sks | COCOK_PERSIS |

## 4. Akurasi per tipe jawaban dan per metode ekstraksi

Rekap ini memisahkan kesalahan karena tipe jawaban sulit dari kesalahan karena jalur ekstraksi tertentu tidak terpakai.

In [4]:
def tabel_rekap(rekap, judul):
    isi = [f"| {judul} | Soal | Persis | Inti | Parafrasa | Longgar | Akurasi inti |",
           "| --- | ---: | ---: | ---: | ---: | ---: | ---: |"]
    for label, sel in rekap.items():
        isi.append(
            f'| {label} | {sel["jumlah"]} | {sel["cocok_persis"]} | {sel["cocok_inti"]} '
            f'| {sel["mirip_parafrasa"]} | {sel["benar_longgar"]} | {sel["akurasi_inti"]:.1%} |'
        )
    display(Markdown("\n".join(isi)))

tabel_rekap(ringkasan["per_tipe_jawaban"], "Tipe jawaban")
tabel_rekap(ringkasan["per_metode"], "Metode")

| Tipe jawaban | Soal | Persis | Inti | Parafrasa | Longgar | Akurasi inti |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| ANGKA | 5 | 3 | 4 | 0 | 4 | 80.0% |
| DEFINISI | 1 | 0 | 0 | 0 | 0 | 0.0% |
| DURASI | 7 | 3 | 7 | 0 | 7 | 100.0% |
| JABATAN | 1 | 1 | 1 | 0 | 1 | 100.0% |
| NILAI_HURUF | 2 | 0 | 0 | 0 | 0 | 0.0% |
| PERSENTASE | 1 | 1 | 1 | 0 | 1 | 100.0% |
| RUMUS | 1 | 0 | 0 | 0 | 0 | 0.0% |
| WAKTU | 2 | 0 | 0 | 2 | 2 | 0.0% |

| Metode | Soal | Persis | Inti | Parafrasa | Longgar | Akurasi inti |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| fallback_kalimat | 4 | 0 | 0 | 0 | 0 | 0.0% |
| linguistik_udpipe | 14 | 6 | 11 | 2 | 13 | 78.6% |
| simbolik | 2 | 2 | 2 | 0 | 2 | 100.0% |

## 5. Analisis kegagalan

Soal yang gagal dipisahkan menurut penyebabnya: bukti rujukan tidak masuk lima besar (masalah retrieval) atau bukti ada tetapi ekstraksi mengambil nilai lain (masalah ekstraksi).

In [5]:
gagal = [baris for baris in baris_hasil if baris["tingkat"] == "TIDAK_COCOK"]
gagal_retrieval = [baris for baris in gagal if not baris["bukti_di_top5"]]
gagal_ekstraksi = [baris for baris in gagal if baris["bukti_di_top5"]]

print(f"Total gagal: {len(gagal)} soal")
print(f"  bukti di luar lima besar: {len(gagal_retrieval)}")
print(f"  bukti ada, jawaban salah: {len(gagal_ekstraksi)}")

for label, kelompok in (("Retrieval", gagal_retrieval), ("Ekstraksi", gagal_ekstraksi)):
    for baris in kelompok:
        print()
        print(f'[{label}] {baris["id_pertanyaan"]} · {baris["pertanyaan"]}')
        print(f'   tipe {baris["tipe_jawaban"]} · {baris["metode"]} · '
              f'peringkat bukti {baris["peringkat_bukti_pertama"]}')
        print(f'   rujukan : {baris["jawaban_rujukan"]}')
        print(f'   sistem  : {baris["jawaban_sistem"]}')

Total gagal: 5 soal
  bukti di luar lima besar: 3
  bukti ada, jawaban salah: 2

[Retrieval] A2-003 · Apa itu skripsi?
   tipe DEFINISI · fallback_kalimat · peringkat bukti 19
   rujukan : Karya ilmiah mahasiswa pendidikan program sarjana yang merupakan wujud dari kajian pengetahuan dan/atau penerapan teknologi berdasarkan kaidah ilmiah dalam bidang studi yang telah dipelajarinya.
   sistem  : Format skripsi disusun menurut dokumen Panduan Skripsi.

[Retrieval] A2-004 · Apa nilai huruf untuk nilai angka 78?
   tipe NILAI_HURUF · fallback_kalimat · peringkat bukti 6
   rujukan : B+
   sistem  : Menurut pedoman penilaian PTIIK, nilai angka (N) dengan rentang 80 < N ≤ 100 dikonversi menjadi nilai huruf A.

[Retrieval] A3-003 · Apa rumus indeks prestasi semester?
   tipe RUMUS · fallback_kalimat · peringkat bukti 25
   rujukan : IP = Σ (K_i × Na_i) / Σ K_i.
   sistem  : Keberhasilan studi mahasiswa dinyatakan dengan Indeks Prestasi (IP).

[Ekstraksi] FARIS-003 · Berapa sks minimal yang har

Lima soal gagal. Tiga soal (A2-003, A2-004, A3-003) sudah bermasalah sejak retrieval karena bukti rujukan berada di peringkat 19, 6, dan 25. Dua soal lain (FARIS-003, SUP-003) buktinya sudah peringkat satu, tetapi ekstraksi mengambil angka lain dari passage yang sama (120 sks untuk soal 138 sks) atau jatuh ke fallback kalimat sehingga jawaban berupa kalimat panjang.

Jalur fallback dipakai empat kali dan belum menghasilkan jawaban benar satu pun, sedangkan jalur simbolik benar pada dua dari dua soal.

## 6. Kebenaran jawaban dikaitkan dengan posisi bukti

Pemisahan ini memperlihatkan bahwa perbaikan pada retrieval berpotensi menaikkan akurasi akhir tanpa mengubah ekstraktor.

In [6]:
display(Markdown(
    f'Bukti di lima besar: **{ringkasan["buktidi_top5"]}/{ringkasan["jumlah_pertanyaan"]}** soal, '
    f'jawaban benar (inti): **{ringkasan["benar_inti_di_top5"]}/{ringkasan["buktidi_top5"]}** '
    f'({ringkasan["akurasi_inti_di_top5"]:.1%}).\n\n'
    f'Bukti di luar lima besar: **{ringkasan["bukti_luar_top5"]}** soal, '
    f'jawaban benar (inti): **{ringkasan["benar_inti_luar_top5"]}/{ringkasan["bukti_luar_top5"]}**.'
))

Bukti di lima besar: **17/20** soal, jawaban benar (inti): **13/17** (76.5%).

Bukti di luar lima besar: **3** soal, jawaban benar (inti): **0/3**.

## 7. Menyimpan hasil penilaian

Keluaran menyimpan ringkasan metrik, rekap per tipe jawaban dan per metode, serta satu baris per soal berisi jawaban rujukan, jawaban sistem, putusan, skor F1, passage sumber, dan peringkat bukti.

In [7]:
from pathlib import Path

path_hasil = Path("output/hasil_evaluasi.json")
path_hasil.parent.mkdir(exist_ok=True)
with path_hasil.open("w", encoding="utf-8") as file:
    json.dump(hasil, file, ensure_ascii=False, indent=2)
    file.write("\n")

print(f"Disimpan di {path_hasil} ({len(baris_hasil)} baris)")

Disimpan di output/hasil_evaluasi.json (20 baris)


## 8. Kesimpulan dan batasan

Pada 20 soal, akurasi ketat 8/20 (40,0%), akurasi inti 13/20 (65,0%), dan akurasi longgar 15/20 (75,0%). Angka ini mengukur kecocokan string dan token, bukan penilaian manusia: jawaban yang benar tetapi disusun berbeda bisa jatuh ke `MIRIP_PARAFRASA`, sedangkan jawaban yang memuat kualifikasi tambahan tetap dihitung benar pada tingkat inti.

Penilaian memakai jawaban rujukan Anggota 5 apa adanya, sehingga kekeliruan pada data rujukan tidak terdeteksi. Tabel penilaian perlu dibaca bersama analisis kegagalan sebelum dipakai sebagai angka akhir laporan.